In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any, List, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

load_dotenv()


True

In [12]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
structured_model = model.with_structured_output(EvaluationSchema)

In [13]:
essay = """
Machine Learning (ML) is a transformative branch of Artificial Intelligence (AI) that empowers computers to learn from data and improve their performance without being explicitly programmed for every specific task. At its core, ML shifts the computing paradigm from rule-based logic—where humans define every "if-then" scenario—to data-driven logic, where algorithms identify patterns and create their own rules [1, 2].
The mechanics of machine learning rely on three primary methodologies: supervised, unsupervised, and reinforcement learning. In supervised learning, algorithms are trained on labeled datasets, meaning the input data is already tagged with the correct answer. This is commonly used for predictive tasks like email spam detection or credit scoring [3]. Unsupervised learning deals with unlabeled data, seeking to discover hidden structures or clusters, such as segmenting customers based on purchasing behavior [1]. Reinforcement learning operates on a system of rewards and penalties, allowing an "agent" to learn the best path through trial and error, a technique foundational to autonomous vehicles and advanced robotics [2, 3].
In 2025, the impact of machine learning is visible across nearly every sector of society. In healthcare, ML models analyze medical imagery with precision rivaling expert radiologists, enabling early detection of diseases like cancer. In finance, it powers high-frequency trading and real-time fraud detection. Moreover, the recent explosion of Generative AI, exemplified by Large Language Models (LLMs), is built entirely on advanced deep learning—a subset of ML that uses neural networks to mimic the human brain’s processing power [4].
"""

In [14]:
# ==== State
class EssayState(TypedDict):
    essay: str
    language_feedback: str 
    analysis_feedback: str
    clarity_feedback: str 
    overall_feedback: str 
    individual_scores: Annotated[List[int], operator.add] 
    # operator.add is a built-in reducer function to concatenate lists 
    # Example: 
    # llm results => [8] + [7] + [9] => reducer result => [8, 7, 9]
    avg_score: int


    

In [15]:
def language_evaluation(state: EssayState) -> Any:
    prompt = """
    Evaluate the language quality of following essay and provide a feedback and assign a score.
    """
    output = structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

def analysis_evaluation(state: EssayState) -> Any:
    prompt = """
    Evaluate the depth of analysis quality of following essay and provide a feedback and assign a score.
    """
    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

def clarity_evaluation(state: EssayState) -> Any:
    prompt = """
    Evaluate the clarity of thought of following essay and provide a feedback and assign a score.
    """
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

def clarity_evaluation(state: EssayState) -> Any:
    prompt = """
    Evaluate the clarity of thought of following essay and provide a feedback and assign a score.
    """
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

def final_evaluation(state: EssayState) -> Any:
    # Summary feedback
    prompt = f"""
    Based on the following feedbacks create a summarized feedback. \n 
    langugage feedback - {state['language_feedback']} \n 
    depth of analysis feedback - {state['analysis_feedback']} \n 
    clarity of thought feedback - {state['clarity_feedback']} 
    """

    final_feedback = model.invoke(prompt).content
    avg_score = sum(state['individual_scores'])/ len(state['individual_scores'])
    # Average score calculation

    return {'final_feedback': final_feedback, 'avg_score': avg_score}



In [16]:
graph = StateGraph(EssayState)
graph.add_node('language_evaluation', language_evaluation)
graph.add_node('analysis_evaluation', analysis_evaluation)
graph.add_node('thought_evaluation', clarity_evaluation)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'language_evaluation')
graph.add_edge(START, 'analysis_evaluation')
graph.add_edge(START, 'thought_evaluation')

graph.add_edge('language_evaluation', 'final_evaluation')
graph.add_edge('analysis_evaluation', 'final_evaluation')
graph.add_edge('thought_evaluation', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

initialState = {
    'essay': essay
}
workflow.invoke(initialState)




{'essay': '\nMachine Learning (ML) is a transformative branch of Artificial Intelligence (AI) that empowers computers to learn from data and improve their performance without being explicitly programmed for every specific task. At its core, ML shifts the computing paradigm from rule-based logic—where humans define every "if-then" scenario—to data-driven logic, where algorithms identify patterns and create their own rules [1, 2].\nThe mechanics of machine learning rely on three primary methodologies: supervised, unsupervised, and reinforcement learning. In supervised learning, algorithms are trained on labeled datasets, meaning the input data is already tagged with the correct answer. This is commonly used for predictive tasks like email spam detection or credit scoring [3]. Unsupervised learning deals with unlabeled data, seeking to discover hidden structures or clusters, such as segmenting customers based on purchasing behavior [1]. Reinforcement learning operates on a system of rewar